# 11.16 字都出現了，為什麼仍可能讀錯順序？


下面的告示先寫「今日休館」，再寫「明日開放」。希望按橫排規則先上後下、每行由左到右，讀成「今日休館／明日開放」。如果先念第二行，字的種類一樣，訊息的順序卻變了。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-11-line-order.svg")))

若只檢查每個字有沒有出現，先念第二行的讀法也會得滿分，所以要整串比較，用 [11.13](https://birdhackor.github.io/tiny-perceptron-vlm/11.13.html) 的 CER：編輯距離是把預測變成正確答案所需的最少插入、刪除或替換次數，CER 再除以正確答案的字數。先看一字替換：把「今天去臺北」當作正確答案，「今天去台北」當作預測結果。


In [ ]:
from tiny_perceptron.natural_concepts import text_error_report

reference = "今天去臺北"  # 正確答案
prediction = "今天去台北"  # 預測結果
report = text_error_report(reference, prediction)  # 先正確答案、後預測
print("完全相同", report["exact"])
print("最少編輯次數", report["edits"])
print("正確答案字數與CER", report["reference_characters"], report["cer"])

`text_error_report(reference, prediction)` 同樣在 [tiny_perceptron/natural_concepts.py](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/tiny_perceptron/natural_concepts.py)：第一個參數是正確答案、第二個是預測，回傳的字典裡，`exact` 是兩串是否完全相同，`edits` 是編輯距離，`reference_characters` 是正確答案的字數，`cer` 是 `edits` 除以這個字數。輸出 False、1，以及 5 與 0.2：正確答案五個字，一字替換。

「臺」和「台」在日常使用裡意思相同，但這個指標比對的是 Unicode 碼點（見 [6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)），不是 UTF-8 bytes：兩字的碼點不同，就算一次替換。本節的中文字都是一字一個碼點，所以碼點數就是字數。若希望把「臺」「台」當成相同，要在看到分數之前訂好文字正規化規則（例如一律把「臺」換成「台」），並同時列出轉換前後的 CER，免得事後挑出對分數有利的規則。

`edits` 是怎麼算出來的？下面是 `text_error_report` 呼叫的 `edit_distance`：


In [ ]:
# 節錄自 tiny_perceptron/natural_concepts.py：edit_distance
def edit_distance(reference, prediction):
    """Levenshtein distance on Python Unicode code points, not byte tokens."""
    previous = list(range(len(prediction) + 1))
    for i, original in enumerate(reference, 1):
        current = [i]
        for j, guessed in enumerate(prediction, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (original != guessed)))
        previous = current
    return previous[-1]

`def` 下一行三個引號包住的英文是說明文字（docstring），不影響計算：Levenshtein distance 就是編輯距離（Levenshtein 是提出這種距離的數學家的姓），比對的是 Python 字串裡的 Unicode 碼點，不是 byte token。

函式本身像在填一張表：第 i 列、第 j 欄記「把預測的前 j 個字改成正確答案的前 i 個字，最少要幾次」，程式一次只保留兩列。`previous = list(range(len(prediction) + 1))` 是第 0 列 `[0, 1, 2, …]`：`range(n)` 依序產生 0 到 n−1，`list(...)` 把它變成清單；意思是正確答案一個字都還沒有時，要把預測的前 j 個字全刪掉，需要 j 次。

`for i, original in enumerate(reference, 1)` 逐一取正確答案的字；`enumerate` 在 [1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html) 用過，第二個參數 1 讓編號從 1 開始。新的一列 `current = [i]` 從 i 開始：預測一個字都沒有時，要補上 i 個字。

內層迴圈對預測的每個字 `guessed` 算一格，用 `min` 從三個數裡取最小：`current[-1] + 1` 是左邊那格再加一次，表示刪掉預測多出的這個字；`previous[j] + 1` 是上面那格再加一次，表示補上預測漏掉的字；`previous[j - 1] + (original != guessed)` 是左上那格，兩字相同時 `(original != guessed)` 是 False、當成 0，不同時是 True、當成 1，表示替換一次。每算完一列，就用 `previous = current` 換成新的一列；最後的 `previous[-1]` 是整串對整串的答案。每格只看左、上、左上三格就夠，因為最省的改法處理兩邊最後一個字（`original` 與 `guessed`），只會是上面這三種動作之一，所以不必試遍所有改法。


In [ ]:
print("多一個 0", edit_distance("1010", "10100"))
print("開頭少了 1", edit_distance("1010", "010"))
print("臺換成台", edit_distance("今天去臺北", "今天去台北"))

三行都印 1：[11.13](https://birdhackor.github.io/tiny-perceptron-vlm/11.13.html) 的兩個例子各只要一次刪除或插入，「臺北」的例子是一次替換，和 `text_error_report` 的 `edits` 相同。若預測是「今天去臺」，只漏掉末尾的「北」，也是一次編輯，分母仍是正確答案的 5 個字，CER＝1/5＝0.2；若預測是「今天去台」，「台」算一次替換，再加上漏掉的「北」，共兩次編輯，CER＝2/5＝0.4。

有沒有文字、逐字內容與讀序，要分別核對。圖上沒有字時，不應憑常見句子補字；字直著寫或分成好幾欄的版面，讀序要另訂規則，不能硬套本例的橫排規則。

練習在最後一段程式加一行 `print("兩行互換", edit_distance("今日休館明日開放", "明日開放今日休館"))`，這裡把兩行直接接成一串字來比。先預測：八個字都在，編輯距離會不會是 0？執行後應看到 6，CER＝6/8＝0.75。6 的來由：兩串逐位對齊時只有第 2、6 位同是「日」，其餘 6 位各換一次就改好；程式回報 6，表示沒有更省的改法。下一節再把有限的物件與短字詞任務安排清楚。

<details>
<summary>回顧與查證</summary>

可回顧：[11.13整串與字元錯誤](https://birdhackor.github.io/tiny-perceptron-vlm/11.13.html)。

</details>
